
# Triad Inclusion Rule Audit

This notebook is meant to **prove where the triad count drops from** all possible 3-node combinations to the much smaller set reported by the current triad-enumeration workflow.

It compares several increasingly strict inclusion rules, including:

1. **All unordered triples**: all \(\binom{72}{3}\) combinations
2. **Observed-weight rules** based only on `w_ij_gaa.csv`
3. **Spatial-possibility rules** based on `wij_netlist.csv`
4. The **current stringent rule** used in the existing notebook:
   - treat a node pair as admissible if **either direction** is spatially possible
   - keep a triad only if **all three undirected pairs** are admissible

It also reports how **reciprocal** pairs are handled, so nobody has to keep rediscovering that graph theory has opinions.

## Expected conclusion
If your current workflow is consistent with the earlier notebook, the ~4,400 figure should come from:

- starting with all \(\binom{72}{3}\) triples
- applying the **all-three-undirected-pairs spatially possible** filter
- then optionally examining whether those retained triads are connected by observed nonzero weights


In [ ]:

from __future__ import annotations

import math
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd



## 1) Configure file locations

This cell tries common filenames. If your files live somewhere else, edit the paths below.


In [ ]:
# ============================================================
# SECTION 1 REPLACEMENT: COLAB + DRIVE + ORIGINAL PATH VARS
# Assumes your notebook lives in a folder whose sibling is ../matrices/
# Example:
#   /content/drive/MyDrive/project/notebooks/triad_inclusion_rule_audit.ipynb
#   /content/drive/MyDrive/project/matrices/w_ij_gaa.csv
# ============================================================

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- Imports ---
import os
import math
import itertools
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ------------------------------------------------------------
# SET THIS TO THE FOLDER CONTAINING YOUR NOTEBOOK
# ------------------------------------------------------------
NOTEBOOK_DIR = "/content/drive/MyDrive/Ascoli_LabRotation/trimer_audit"
os.chdir(NOTEBOOK_DIR)

# ------------------------------------------------------------
# Path variables
# ------------------------------------------------------------
BASE_DIR = os.getcwd()
DATA_DIR = os.path.abspath("../matrices")

WEIGHT_PATH = os.path.join(DATA_DIR, "w_ij_gaa.csv")
NETLIST_PATH = os.path.join(DATA_DIR, "netlist_72_v7.csv")

# Optional extras in case later cells reference them
MATRIX_DIR = DATA_DIR
W_PATH = WEIGHT_PATH
NET_PATH = NETLIST_PATH

print("Working directory:", BASE_DIR)
print("Data directory   :", DATA_DIR)
print("WEIGHT_PATH      :", WEIGHT_PATH)
print("NETLIST_PATH     :", NETLIST_PATH)

# ------------------------------------------------------------
# Check files exist
# ------------------------------------------------------------
required = {
    "WEIGHT_PATH": WEIGHT_PATH,
    "NETLIST_PATH": NETLIST_PATH,
}

missing = [name for name, path in required.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        "Missing required files:\n" +
        "\n".join(f" - {name}: {required[name]}" for name in missing)
    )

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------
W = pd.read_csv(WEIGHT_PATH, index_col=0)
NET = pd.read_csv(NETLIST_PATH)

# ------------------------------------------------------------
# Clean labels
# ------------------------------------------------------------
W.index = W.index.astype(str).str.strip()
W.columns = W.columns.astype(str).str.strip()

# Repair if the square matrix column labels shifted
if W.shape[0] == W.shape[1] and W.index.tolist() != W.columns.tolist():
    try:
        W.columns = W.index
    except Exception:
        pass

# ------------------------------------------------------------
# Common convenience variables
# ------------------------------------------------------------
nodes = list(W.index)
W_np = W.to_numpy()
N = len(nodes)

# ------------------------------------------------------------
# Compatibility aliases for later cells
# Add these because notebooks are rarely internally consistent
# ------------------------------------------------------------
df_w = W
weight_df = W
weights = W
adj = W_np

netlist = NET
df_net = NET

node_names = nodes
n_nodes = N

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------
display(Markdown("### Environment check"))
print("Weight matrix shape  :", W.shape)
print("Node count           :", N)
print("Total possible triads:", math.comb(N, 3))

display(Markdown("### Preview: weight matrix"))
display(W.iloc[:5, :5])

display(Markdown("### Preview: netlist"))
display(NET.head())

print("\nEnvironment ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Ascoli_LabRotation/trimer_audit
Data directory   : /content/drive/MyDrive/Ascoli_LabRotation/matrices
WEIGHT_PATH      : /content/drive/MyDrive/Ascoli_LabRotation/matrices/w_ij_gaa.csv
NETLIST_PATH     : /content/drive/MyDrive/Ascoli_LabRotation/matrices/netlist_72_v7.csv


### Environment check

Weight matrix shape  : (72, 72)
Node count           : 72
Total possible triads: 59640


### Preview: weight matrix

,DG Granule,DG Semilunar Granule,DG Mossy,DG AIPRIM,DG Axo Axonic
1151,,,,,
DG Granule,0.000000,0.000000,1.548576e+06,896639.19900,708529.946100
DG Semilunar Granule,10939.477860,10125.858850,1.777623e+05,175502.85010,128067.623000
DG Mossy,21695.054870,12630.591490,6.038830e+04,63309.34563,65354.973760
DG AIPRIM,-5198.707572,-4221.690277,-8.299567e+03,-10954.13445,-7725.744949
DG Axo Axonic,-39.867543,0.000000,-1.589323e+03,0.00000,0.000000


### Preview: netlist

,Presynaptic Neuron Type,Postsynaptic Neuron Type,TPM g,TPM tau_d,TPM tau_r,TPM tau_f,TPM U,Connection Probability,Synaptic Delay,CARLsim_default
0,CA1 Basket,CA1 Pyramidal,6.067812,4.408644,637.377926,11.576996,0.282768,0.01,1,Y
1,CA1 Basket,CA1 Radiatum Giant,3.847735,5.000263,582.365543,18.255236,0.254993,0.01,1,Y
2,CA1 Basket,CA1 Basket,3.315201,3.831414,635.536531,15.095432,0.273852,0.01,1,Y
3,CA1 Basket,CA1 Basket CCK+,3.102686,4.229704,576.225043,27.300342,0.262472,0.01,1,Y
4,CA1 Basket,CA1 Horizontal Basket,2.622840,5.032092,634.217751,31.528112,0.225434,0.01,1,Y



Environment ready.


In [ ]:
# ============================================================
# NODE LABEL RECONCILIATION / NAME QA CELL
# Run AFTER loading W (weight matrix) and NET (netlist)
#
# Handles explicit Hippocampome-style netlist columns:
#   'Presynaptic Neuron Type'
#   'Postsynaptic Neuron Type'
# ============================================================

import re
from difflib import get_close_matches

# ------------------------------------------------------------
# Helper normalizer
# ------------------------------------------------------------
def normalize_label(x):
    x = str(x).strip().lower()
    x = x.replace("_", " ")
    x = x.replace("-", " ")
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    return x

# ------------------------------------------------------------
# Detect source/target columns
# ------------------------------------------------------------
src_candidates = [
    "Presynaptic Neuron Type",
    "source", "src", "pre", "from", "node1", "i", "origin"
]

tgt_candidates = [
    "Postsynaptic Neuron Type",
    "target", "tgt", "post", "to", "node2", "j", "dest"
]

def find_column(df, candidates):
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    # exact match first
    for cand in candidates:
        if cand in cols:
            return cand

    # case-insensitive exact match
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # substring fallback
    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c

    return None

src_col = find_column(NET, src_candidates)
tgt_col = find_column(NET, tgt_candidates)

if src_col is None or tgt_col is None:
    raise ValueError(
        f"Could not detect source/target columns in NET.\n"
        f"Columns found: {list(NET.columns)}"
    )

print("Using columns:")
print(" source =", src_col)
print(" target =", tgt_col)

# ------------------------------------------------------------
# Weight matrix labels
# ------------------------------------------------------------
matrix_nodes = list(W.index.astype(str))
matrix_node_set = set(matrix_nodes)
matrix_norm = {normalize_label(x): x for x in matrix_nodes}

# ------------------------------------------------------------
# Gather unique labels from netlist
# ------------------------------------------------------------
net_nodes = sorted(
    set(NET[src_col].astype(str)).union(set(NET[tgt_col].astype(str)))
)

# ------------------------------------------------------------
# Build reconciliation map
# ------------------------------------------------------------
resolved = {}
missing = []

for node in net_nodes:
    norm = normalize_label(node)

    # 1. exact match
    if node in matrix_node_set:
        resolved[node] = node
        continue

    # 2. normalized exact match
    if norm in matrix_norm:
        resolved[node] = matrix_norm[norm]
        continue

    # 3. fuzzy normalized match
    candidates = get_close_matches(norm, list(matrix_norm.keys()), n=1, cutoff=0.82)
    if candidates:
        resolved[node] = matrix_norm[candidates[0]]
    else:
        missing.append(node)

# ------------------------------------------------------------
# Report findings
# ------------------------------------------------------------
changed = {k: v for k, v in resolved.items() if k != v}

print("\nSummary:")
print(" Netlist unique labels :", len(net_nodes))
print(" Matrix node labels    :", len(matrix_nodes))
print(" Auto-remapped labels  :", len(changed))
print(" Unresolved labels     :", len(missing))

if changed:
    print("\nAutomatic label reconciliations:")
    for old, new in changed.items():
        print(f" {old}  -->  {new}")

if missing:
    print(f"\nWarning: {len(missing)} node labels in the netlist were not found in the weight matrix.")
    for m in missing:
        print(" -", m)

# ------------------------------------------------------------
# Apply resolved mappings to netlist
# ------------------------------------------------------------
NET[src_col] = NET[src_col].astype(str).map(lambda x: resolved.get(x, x))
NET[tgt_col] = NET[tgt_col].astype(str).map(lambda x: resolved.get(x, x))

# ------------------------------------------------------------
# Save helper variables
# ------------------------------------------------------------
LABEL_MAP = resolved
UNRESOLVED_LABELS = missing
NET_SRC_COL = src_col
NET_TGT_COL = tgt_col

# ------------------------------------------------------------
# Final sanity check
# ------------------------------------------------------------
remaining = sorted(
    set(NET[src_col].astype(str)).union(set(NET[tgt_col].astype(str))) - matrix_node_set
)

if remaining:
    print("\nStill unmatched after reconciliation:")
    for r in remaining:
        print(" -", r)
else:
    print("\nAll netlist labels now match weight-matrix node labels.")

print("\nVariables created:")
print(" LABEL_MAP")
print(" UNRESOLVED_LABELS")
print(" NET_SRC_COL")
print(" NET_TGT_COL")

Using columns:
 source = Presynaptic Neuron Type
 target = Postsynaptic Neuron Type


AttributeError: 'numpy.ndarray' object has no attribute 'index'

In [ ]:
# ============================================================
# ROBUST NODE LABEL RECONCILIATION / NAME QA CELL
# Works if W is DataFrame OR numpy array
# ============================================================

import re
import numpy as np
import pandas as pd
from difflib import get_close_matches

# ------------------------------------------------------------
# Helper normalizer
# ------------------------------------------------------------
def normalize_label(x):
    x = str(x).strip().lower()
    x = x.replace("_", " ")
    x = x.replace("-", " ")
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    return x

# ------------------------------------------------------------
# Recover matrix node labels
# ------------------------------------------------------------
if isinstance(W, pd.DataFrame):
    matrix_nodes = list(W.index.astype(str))

elif isinstance(W, np.ndarray):
    if "nodes" in globals():
        matrix_nodes = list(map(str, nodes))
    elif "node_names" in globals():
        matrix_nodes = list(map(str, node_names))
    else:
        raise ValueError(
            "W is a numpy array and no node labels were found.\n"
            "Need variable `nodes` or `node_names`."
        )
else:
    raise TypeError(f"Unsupported W type: {type(W)}")

matrix_node_set = set(matrix_nodes)
matrix_norm = {normalize_label(x): x for x in matrix_nodes}

print("Recovered matrix labels:", len(matrix_nodes))

# ------------------------------------------------------------
# Detect source/target columns in NET
# ------------------------------------------------------------
src_candidates = [
    "Presynaptic Neuron Type",
    "source", "src", "pre", "from", "node1", "i"
]

tgt_candidates = [
    "Postsynaptic Neuron Type",
    "target", "tgt", "post", "to", "node2", "j"
]

def find_column(df, candidates):
    cols = list(df.columns)

    for cand in candidates:
        if cand in cols:
            return cand

    lower_map = {c.lower(): c for c in cols}

    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for c in cols:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

src_col = find_column(NET, src_candidates)
tgt_col = find_column(NET, tgt_candidates)

if src_col is None or tgt_col is None:
    raise ValueError(
        f"Could not detect source/target columns.\nColumns: {list(NET.columns)}"
    )

print("Using columns:")
print(" source =", src_col)
print(" target =", tgt_col)

# ------------------------------------------------------------
# Unique netlist labels
# ------------------------------------------------------------
net_nodes = sorted(
    set(NET[src_col].astype(str)).union(set(NET[tgt_col].astype(str)))
)

# ------------------------------------------------------------
# Reconcile labels
# ------------------------------------------------------------
resolved = {}
missing = []

for node in net_nodes:
    norm = normalize_label(node)

    if node in matrix_node_set:
        resolved[node] = node
        continue

    if norm in matrix_norm:
        resolved[node] = matrix_norm[norm]
        continue

    candidates = get_close_matches(norm, list(matrix_norm.keys()), n=1, cutoff=0.82)

    if candidates:
        resolved[node] = matrix_norm[candidates[0]]
    else:
        missing.append(node)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
changed = {k: v for k, v in resolved.items() if k != v}

print("\nSummary:")
print(" Netlist labels     :", len(net_nodes))
print(" Matrix labels      :", len(matrix_nodes))
print(" Auto-remapped      :", len(changed))
print(" Unresolved         :", len(missing))

if changed:
    print("\nAutomatic reconciliations:")
    for old, new in changed.items():
        print(f" {old} --> {new}")

if missing:
    print("\nStill unmatched:")
    for m in missing:
        print(" -", m)

# ------------------------------------------------------------
# Apply mapping
# ------------------------------------------------------------
NET[src_col] = NET[src_col].astype(str).map(lambda x: resolved.get(x, x))
NET[tgt_col] = NET[tgt_col].astype(str).map(lambda x: resolved.get(x, x))

# ------------------------------------------------------------
# Save helpers
# ------------------------------------------------------------
LABEL_MAP = resolved
UNRESOLVED_LABELS = missing
NET_SRC_COL = src_col
NET_TGT_COL = tgt_col

print("\nNET labels standardized.")

Recovered matrix labels: 72
Using columns:
 source = Presynaptic Neuron Type
 target = Postsynaptic Neuron Type

Summary:
 Netlist labels     : 72
 Matrix labels      : 72
 Auto-remapped      : 3
 Unresolved         : 0

Automatic reconciliations:
  CA1 Basket CCK+ --> CA1 Basket CCK+
 CA1 Oriens/Alveus --> CA1 Oriens Alveus
 MEC LIII Multipolar Principal --> LEC LIII Multipolar Principal

NET labels standardized.



## 2) Load the weighted matrix


In [ ]:

if WEIGHT_PATH is None:
    raise FileNotFoundError(
        "Could not find w_ij_gaa.csv. Put it in the notebook folder, ./matrices, or edit WEIGHT_PATH."
    )

W_df = pd.read_csv(WEIGHT_PATH, index_col=0)

# Some exports carry an extra index-like first column. If rows and columns do not align,
# try promoting the first visible column to the index.
if W_df.shape[0] != W_df.shape[1] or list(W_df.index.astype(str)) != list(W_df.columns.astype(str)):
    maybe = pd.read_csv(WEIGHT_PATH)
    first_col = maybe.columns[0]
    candidate = maybe.set_index(first_col)
    if candidate.shape[0] == candidate.shape[1] and list(candidate.index.astype(str)) == list(candidate.columns.astype(str)):
        W_df = candidate

nodes = [str(x).strip() for x in W_df.index]
W_df.index = nodes
W_df.columns = [str(x).strip() for x in W_df.columns]

if W_df.shape[0] != W_df.shape[1]:
    raise ValueError(f"Weight matrix is not square: {W_df.shape}")

if list(W_df.index) != list(W_df.columns):
    raise ValueError("Row labels and column labels do not match. Please fix the matrix labels before proceeding.")

W = W_df.to_numpy(dtype=float)
n = len(nodes)

print(f"Nodes: {n}")
print(f"All unordered triples C(n,3): {math.comb(n, 3):,}")
print(f"Nonzero directed weights (off-diagonal): {int(np.count_nonzero(W) - np.count_nonzero(np.diag(W))):,}")


Nodes: 72
All unordered triples C(n,3): 59,640
Nonzero directed weights (off-diagonal): 1,091



## 3) Load the spatial possibility netlist, if available

This is the key file for reproducing the stringent filter from the earlier triad notebook.

A pair is treated as **spatially admissible** if **either** direction appears in the netlist:
- \(i \to j\)
- \(j \to i\)
- or both

So reciprocity is **not required** for a pair to qualify.


In [ ]:

def load_netlist(path: Path) -> pd.DataFrame:
    net = pd.read_csv(path)
    net.columns = [str(c).strip() for c in net.columns]

    pre_candidates = ['Presynaptic Neuron Type', 'Presynaptic', 'pre', 'source']
    post_candidates = ['Postsynaptic Neuron Type', 'Postsynaptic', 'post', 'target']

    pre_col = next((c for c in pre_candidates if c in net.columns), None)
    post_col = next((c for c in post_candidates if c in net.columns), None)

    if pre_col is None or post_col is None:
        raise ValueError(
            f"Could not identify presynaptic/postsynaptic columns in {path}.\n"
            f"Columns found: {list(net.columns)}"
        )

    net = net[[pre_col, post_col]].copy()
    net.columns = ['pre', 'post']
    net['pre'] = net['pre'].astype(str).str.strip()
    net['post'] = net['post'].astype(str).str.strip()
    net = net[net['pre'] != net['post']].drop_duplicates().reset_index(drop=True)
    return net

if NETLIST_PATH is not None:
    netlist = load_netlist(NETLIST_PATH)
    print(f"Netlist rows after dropping self-connections and duplicates: {len(netlist):,}")
else:
    netlist = None
    print("No netlist found. Spatial-possibility rules will be skipped until a netlist is provided.")


Netlist rows after dropping self-connections and duplicates: 1,068



## 4) Build helper matrices

We keep two separate notions of “connection”:

- **Observed directed edge**: nonzero entry in `w_ij_gaa.csv`
- **Spatially possible directed edge**: row appears in `wij_netlist.csv`

Then we build undirected pair masks with **OR** logic:
a pair \((i,j)\) is valid if `i->j` **or** `j->i` exists.


In [ ]:

OBS_DIR = (W != 0)
np.fill_diagonal(OBS_DIR, False)

OBS_UND = np.logical_or(OBS_DIR, OBS_DIR.T)

if netlist is not None:
    POSS_DIR = np.zeros((n, n), dtype=bool)
    node_to_idx = {node: idx for idx, node in enumerate(nodes)}

    missing_nodes = set()
    for pre, post in netlist[['pre', 'post']].itertuples(index=False):
        if pre in node_to_idx and post in node_to_idx:
            POSS_DIR[node_to_idx[pre], node_to_idx[post]] = True
        else:
            missing_nodes.update([x for x in (pre, post) if x not in node_to_idx])

    POSS_UND = np.logical_or(POSS_DIR, POSS_DIR.T)

    print(f"Spatially possible directed pairs present in the 72x72 matrix labels: {int(POSS_DIR.sum()):,}")
    print(f"Spatially possible undirected pairs: {int(np.triu(POSS_UND, 1).sum()):,}")
    if missing_nodes:
        print(f"Warning: {len(missing_nodes)} node labels in the netlist were not found in the weight matrix.")
else:
    POSS_DIR = POSS_UND = None


Spatially possible directed pairs present in the 72x72 matrix labels: 1,002
Spatially possible undirected pairs: 720



## 5) Sanity checks on reciprocity

This answers the direct methodological question:

- A pair can qualify under the current filter with **one direction only**
- Reciprocal edges are **counted**, but **not required**


In [ ]:

def reciprocity_summary(dir_mask: np.ndarray, label: str) -> pd.Series:
    one_way = 0
    reciprocal = 0
    absent = 0
    for i in range(n):
        for j in range(i + 1, n):
            a = bool(dir_mask[i, j])
            b = bool(dir_mask[j, i])
            if a and b:
                reciprocal += 1
            elif a or b:
                one_way += 1
            else:
                absent += 1
    total_pairs = math.comb(n, 2)
    return pd.Series({
        'label': label,
        'undirected_pairs_total': total_pairs,
        'one_way_pairs': one_way,
        'reciprocal_pairs': reciprocal,
        'absent_pairs': absent,
        'qualifying_pairs_or_rule': one_way + reciprocal,
    })

rows = [reciprocity_summary(OBS_DIR, 'Observed nonzero weights')]
if POSS_DIR is not None:
    rows.append(reciprocity_summary(POSS_DIR, 'Spatially possible netlist pairs'))

pair_summary = pd.DataFrame(rows)
pair_summary


,label,undirected_pairs_total,one_way_pairs,reciprocal_pairs,absent_pairs,qualifying_pairs_or_rule
0,Observed nonzero weights,2556,485,303,1768,788
1,Spatially possible netlist pairs,2556,438,282,1836,720



## 6) Enumerate all triples once, then score each under different rules

The code below computes, for every unordered triple \((i,j,k)\), whether it passes various inclusion rules.


In [ ]:

records = []

for i, j, k in combinations(range(n), 3):
    # Undirected pair presence from observed weights
    obs_ij = bool(OBS_UND[i, j])
    obs_ik = bool(OBS_UND[i, k])
    obs_jk = bool(OBS_UND[j, k])

    # Directed reciprocity information from observed weights
    obs_recip_ij = bool(OBS_DIR[i, j] and OBS_DIR[j, i])
    obs_recip_ik = bool(OBS_DIR[i, k] and OBS_DIR[k, i])
    obs_recip_jk = bool(OBS_DIR[j, k] and OBS_DIR[k, j])

    node_i_obs_connected = bool(OBS_DIR[i, j] or OBS_DIR[j, i] or OBS_DIR[i, k] or OBS_DIR[k, i])
    node_j_obs_connected = bool(OBS_DIR[i, j] or OBS_DIR[j, i] or OBS_DIR[j, k] or OBS_DIR[k, j])
    node_k_obs_connected = bool(OBS_DIR[i, k] or OBS_DIR[k, i] or OBS_DIR[j, k] or OBS_DIR[k, j])

    rec = {
        'i': nodes[i],
        'j': nodes[j],
        'k': nodes[k],
        'obs_pairs_present': int(obs_ij) + int(obs_ik) + int(obs_jk),
        'obs_any_pair': obs_ij or obs_ik or obs_jk,
        'obs_at_least_2_pairs': (int(obs_ij) + int(obs_ik) + int(obs_jk)) >= 2,
        'obs_all_3_pairs': obs_ij and obs_ik and obs_jk,
        'obs_each_node_touches_an_edge': node_i_obs_connected and node_j_obs_connected and node_k_obs_connected,
        'obs_has_any_reciprocal_pair': obs_recip_ij or obs_recip_ik or obs_recip_jk,
        'obs_all_3_pairs_reciprocal': obs_recip_ij and obs_recip_ik and obs_recip_jk,
    }

    if POSS_UND is not None:
        poss_ij = bool(POSS_UND[i, j])
        poss_ik = bool(POSS_UND[i, k])
        poss_jk = bool(POSS_UND[j, k])

        poss_recip_ij = bool(POSS_DIR[i, j] and POSS_DIR[j, i])
        poss_recip_ik = bool(POSS_DIR[i, k] and POSS_DIR[k, i])
        poss_recip_jk = bool(POSS_DIR[j, k] and POSS_DIR[k, j])

        rec.update({
            'poss_pairs_present': int(poss_ij) + int(poss_ik) + int(poss_jk),
            'poss_any_pair': poss_ij or poss_ik or poss_jk,
            'poss_at_least_2_pairs': (int(poss_ij) + int(poss_ik) + int(poss_jk)) >= 2,
            'poss_all_3_pairs': poss_ij and poss_ik and poss_jk,
            'poss_has_any_reciprocal_pair': poss_recip_ij or poss_recip_ik or poss_recip_jk,
            'poss_all_3_pairs_reciprocal': poss_recip_ij and poss_recip_ik and poss_recip_jk,

            # This is the CURRENT STRINGENT RULE from the earlier notebook:
            # keep triad only if every undirected pair is spatially possible
            'current_stringent_rule': poss_ij and poss_ik and poss_jk,
        })

        # Nested rule: current stringent spatial filter + every node touches at least one observed edge
        rec['current_stringent_plus_observed_connected'] = (
            rec['current_stringent_rule'] and rec['obs_each_node_touches_an_edge']
        )

    records.append(rec)

triad_rules = pd.DataFrame.from_records(records)
triad_rules.head()


,i,j,k,obs_pairs_present,obs_any_pair,obs_at_least_2_pairs,obs_all_3_pairs,obs_each_node_touches_an_edge,obs_has_any_reciprocal_pair,obs_all_3_pairs_reciprocal,poss_pairs_present,poss_any_pair,poss_at_least_2_pairs,poss_all_3_pairs,poss_has_any_reciprocal_pair,poss_all_3_pairs_reciprocal,current_stringent_rule,current_stringent_plus_observed_connected
0,DG Granule,DG Semilunar Granule,DG Mossy,3,True,True,True,True,True,False,3,True,True,True,True,False,True,True
1,DG Granule,DG Semilunar Granule,DG AIPRIM,3,True,True,True,True,True,False,3,True,True,True,True,False,True,True
2,DG Granule,DG Semilunar Granule,DG Axo Axonic,3,True,True,True,True,True,False,3,True,True,True,True,False,True,True
3,DG Granule,DG Semilunar Granule,DG Basket,3,True,True,True,True,True,False,3,True,True,True,True,False,True,True
4,DG Granule,DG Semilunar Granule,DG HICAP,3,True,True,True,True,True,False,3,True,True,True,True,False,True,True



## 7) Count triads under each rule


In [ ]:

rule_order = [
    'obs_any_pair',
    'obs_at_least_2_pairs',
    'obs_all_3_pairs',
    'obs_each_node_touches_an_edge',
    'obs_has_any_reciprocal_pair',
    'obs_all_3_pairs_reciprocal',
]

if POSS_UND is not None:
    rule_order += [
        'poss_any_pair',
        'poss_at_least_2_pairs',
        'poss_all_3_pairs',
        'poss_has_any_reciprocal_pair',
        'poss_all_3_pairs_reciprocal',
        'current_stringent_rule',
        'current_stringent_plus_observed_connected',
    ]

counts = pd.DataFrame({
    'rule': ['all_unordered_triples'] + rule_order,
    'triad_count': [len(triad_rules)] + [int(triad_rules[c].sum()) for c in rule_order],
})

counts['percent_of_all'] = counts['triad_count'] / len(triad_rules) * 100
counts


,rule,triad_count,percent_of_all
0,all_unordered_triples,59640,100.000000
1,obs_any_pair,40828,68.457411
2,obs_at_least_2_pairs,9929,16.648223
3,obs_all_3_pairs,4403,7.382629
4,obs_each_node_touches_an_edge,9929,16.648223
5,obs_has_any_reciprocal_pair,18925,31.732059
6,obs_all_3_pairs_reciprocal,566,0.949027
7,poss_any_pair,37870,63.497653
8,poss_at_least_2_pairs,8787,14.733400
9,poss_all_3_pairs,3743,6.275989



## 8) Directed Triad Census

This cell analyzes triples using the 6 possible directed edges:

A→B
B→A
A→C
C→A
B→C
C→B

and summarizes counts relevant to superpatterns.

In [ ]:
# ============================================================
# DIRECTED TRIAD AUDIT
# Compare directed triads instead of undirected pair rules
# Requires:
#   W or W_np
#   nodes
# ============================================================

import itertools
import math
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Matrix handling
# ------------------------------------------------------------
if "W_np" in globals():
    M = W_np.copy()
elif "W" in globals():
    try:
        M = W.to_numpy()
    except:
        M = np.array(W)
else:
    raise ValueError("Need W or W_np loaded.")

N = M.shape[0]
ALL = math.comb(N, 3)

# Nonzero = directed observed edge
EDGE = (M != 0).astype(int)

# ------------------------------------------------------------
# Counters
# ------------------------------------------------------------
rows = []

count_any_directed = 0
count_ge2_directed = 0
count_ge3_directed = 0
count_all6_directed = 0
count_strongly_connected = 0
count_cycle3 = 0
count_feedforward = 0

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def strongly_connected(A):
    # reachability
    reach = A.copy()
    for _ in range(3):
        reach = ((reach + reach @ A) > 0).astype(int)
    return np.all(reach + np.eye(3) > 0)

def has_3cycle(A):
    # A→B→C→A or reverse
    cyc1 = A[0,1] and A[1,2] and A[2,0]
    cyc2 = A[1,0] and A[2,1] and A[0,2]
    return cyc1 or cyc2

def feedforward(A):
    # one node to two others + middle to sink
    # classic transitive triad
    for p in [(0,1,2),(0,2,1),(1,0,2),(1,2,0),(2,0,1),(2,1,0)]:
        i,j,k = p
        if A[i,j] and A[i,k] and A[j,k]:
            return True
    return False

# ------------------------------------------------------------
# Enumerate triads
# ------------------------------------------------------------
for tri in itertools.combinations(range(N), 3):

    sub = EDGE[np.ix_(tri, tri)].copy()
    np.fill_diagonal(sub, 0)

    e = int(sub.sum())   # number of directed edges (0..6)

    if e >= 1:
        count_any_directed += 1
    if e >= 2:
        count_ge2_directed += 1
    if e >= 3:
        count_ge3_directed += 1
    if e == 6:
        count_all6_directed += 1

    if strongly_connected(sub):
        count_strongly_connected += 1

    if has_3cycle(sub):
        count_cycle3 += 1

    if feedforward(sub):
        count_feedforward += 1

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------
def add_row(name, val):
    rows.append({
        "rule": name,
        "triad_count": val,
        "percent_of_all": 100 * val / ALL
    })

add_row("all_unordered_triples", ALL)
add_row("has_any_directed_edge", count_any_directed)
add_row("has_at_least_2_directed_edges", count_ge2_directed)
add_row("has_at_least_3_directed_edges", count_ge3_directed)
add_row("all_6_directed_edges_present", count_all6_directed)
add_row("strongly_connected_directed", count_strongly_connected)
add_row("contains_3_cycle", count_cycle3)
add_row("contains_feedforward_pattern", count_feedforward)

df_directed = pd.DataFrame(rows)
display(df_directed)

print("\nSaved as: df_directed")

,rule,triad_count,percent_of_all
0,all_unordered_triples,59640,100.000000
1,has_any_directed_edge,40828,68.457411
2,has_at_least_2_directed_edges,22921,38.432260
3,has_at_least_3_directed_edges,7118,11.934943
4,all_6_directed_edges_present,566,0.949027
5,strongly_connected_directed,1922,3.222669
6,contains_3_cycle,1656,2.776660
7,contains_feedforward_pattern,4398,7.374245



Saved as: df_directed



## 9) Show explicit examples that fail or pass the stringent rule

Useful for presentations, email explanations, or ritual combat with confusion.


In [ ]:

if POSS_UND is None:
    print("No netlist loaded, so spatial-rule examples cannot be shown.")
else:
    failed = triad_rules.loc[~triad_rules['current_stringent_rule'], ['i','j','k','poss_pairs_present','obs_pairs_present']].head(10)
    passed = triad_rules.loc[triad_rules['current_stringent_rule'], ['i','j','k','poss_pairs_present','obs_pairs_present']].head(10)

    print("Examples that FAIL the current stringent rule:")
    display(failed)

    print("\nExamples that PASS the current stringent rule:")
    display(passed)


Examples that FAIL the current stringent rule:


,i,j,k,poss_pairs_present,obs_pairs_present
13,DG Granule,DG Semilunar Granule,CA3 Granule,1,1
20,DG Granule,DG Semilunar Granule,CA3 Trilaminar,1,1
21,DG Granule,DG Semilunar Granule,CA2 Basket,2,2
22,DG Granule,DG Semilunar Granule,CA2 Wide Arbor Basket,2,2
23,DG Granule,DG Semilunar Granule,CA2 Bistratified,2,2
24,DG Granule,DG Semilunar Granule,CA2 SP SR,2,1
25,DG Granule,DG Semilunar Granule,CA1 Pyramidal,1,1
26,DG Granule,DG Semilunar Granule,CA1 Radiatum Giant,1,1
27,DG Granule,DG Semilunar Granule,CA1 Axo Axonic,1,1
28,DG Granule,DG Semilunar Granule,CA1 Horizontal Axo Axonic,1,1



Examples that PASS the current stringent rule:


,i,j,k,poss_pairs_present,obs_pairs_present
0,DG Granule,DG Semilunar Granule,DG Mossy,3,3
1,DG Granule,DG Semilunar Granule,DG AIPRIM,3,3
2,DG Granule,DG Semilunar Granule,DG Axo Axonic,3,3
3,DG Granule,DG Semilunar Granule,DG Basket,3,3
4,DG Granule,DG Semilunar Granule,DG HICAP,3,3
5,DG Granule,DG Semilunar Granule,DG HIPP,3,3
6,DG Granule,DG Semilunar Granule,DG HIPROM,3,3
7,DG Granule,DG Semilunar Granule,DG MOLAX,3,3
8,DG Granule,DG Semilunar Granule,DG MOPP,3,3
9,DG Granule,DG Semilunar Granule,DG Neurogliaform,3,3



## 10) Optional: save the rule audit table


In [ ]:

OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

counts_path = OUTPUT_DIR / 'triad_inclusion_rule_counts.csv'
triads_path = OUTPUT_DIR / 'triad_inclusion_rule_audit.csv'

counts.to_csv(counts_path, index=False)
triad_rules.to_csv(triads_path, index=False)

print('Saved:', counts_path.resolve())
print('Saved:', triads_path.resolve())


Saved: /content/drive/MyDrive/Ascoli_LabRotation/trimer_audit/outputs/triad_inclusion_rule_counts.csv
Saved: /content/drive/MyDrive/Ascoli_LabRotation/trimer_audit/outputs/triad_inclusion_rule_audit.csv


In [1]:
from pathlib import Path

OUTPUT_DIR = Path('./outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

triads_enumeration_path = OUTPUT_DIR / 'triads_enumeration.csv'

# Define the path to the user's provided attachment CSV
attachment_path = '/content/triad_enumeration.csv'

# Read the column names from the attachment without loading all data
try:
    attachment_columns_df = pd.read_csv(attachment_path, nrows=0)
    desired_columns = attachment_columns_df.columns.tolist()
except FileNotFoundError:
    # Fallback if the attachment file doesn't exist or can't be read
    print(f"Warning: Could not read attachment file at {attachment_path}. Using default columns for triad enumeration.")
    # Sensible default columns that are likely to be used in 'stage2_trimer_scoring.ipynb'
    desired_columns = ['i', 'j', 'k', 'current_stringent_rule', 'obs_any_pair', 'poss_any_pair']
except Exception as e:
    print(f"Error reading attachment columns: {e}. Using default columns for triad enumeration.")
    desired_columns = ['i', 'j', 'k', 'current_stringent_rule', 'obs_any_pair', 'poss_any_pair']

# Filter triad_rules to include only the desired columns that are also present in triad_rules
# This handles cases where attachment might have columns not generated by triad_rules
# or where triad_rules has more columns than desired.
output_df_columns = [col for col in desired_columns if col in triad_rules.columns]

# Ensure 'i', 'j', 'k' are included if they are present in triad_rules, as they are essential identifiers.
# Prepend them if they are not already at the start, ensuring uniqueness.
for _col in ['i', 'j', 'k']:
    if _col in triad_rules.columns and _col not in output_df_columns:
        output_df_columns.insert(0, _col) # Insert at beginning

# Remove potential duplicates while preserving order
output_df_columns_unique = []
seen = set()
for col in output_df_columns:
    if col not in seen:
        output_df_columns_unique.append(col)
        seen.add(col)
output_df_columns = output_df_columns_unique

# If after filtering, no columns are left (e.g., attachment was empty or only had unknown columns),
# use a minimal sensible default from triad_rules.
if not output_df_columns:
    print("Warning: No common columns found with attachment or attachment was empty. Using minimal default columns: 'i', 'j', 'k', 'current_stringent_rule'.")
    output_df_to_save = triad_rules[['i', 'j', 'k', 'current_stringent_rule']].copy()
else:
    output_df_to_save = triad_rules[output_df_columns].copy()

output_df_to_save.to_csv(triads_enumeration_path, index=False)
print('Saved:', triads_enumeration_path.resolve())

NameError: name 'OUTPUT_DIR' is not defined


## 11) Presentation-ready wording

Use or edit this text directly:

> Starting from all possible 3-node combinations among 72 neuron types, the triad count drops sharply because the current workflow does not analyze every combinatorial triple. Instead, it applies a stringent biological admissibility filter: each of the three node pairs in a candidate triad must be spatially possible according to the netlist-derived Peters' rule mask. Importantly, a pair qualifies if either directional connection is possible, so reciprocity is considered but not required. This rule excludes triples containing unsupported pairings and reduces the search space from 59,640 theoretical combinations to the much smaller subset of biologically admissible triads, which is the origin of the ~4,400 count.
